# Human review gates for Claude agents with Gatewerk MCP

This notebook shows how to build a Claude agent that submits work for human review
using Gatewerk, waits for a human decision, and then acts on the outcome — including
any payload edits the reviewer made.

## What this demonstrates

1. Define a `create_gatewerk_review` tool so Claude can open a review gate.
2. Let Claude call the tool in a standard tool-use loop.
3. Execute the HTTP call against your local Gatewerk instance.
4. Poll `GET /reviews/:id` until `status` is `decided`.
5. Branch on the `decision` field (`approved` / `rejected` / `edited`) — feeding
   `approved_value` back to Claude as context for the next step.

## Prerequisites

| Requirement | How to get it |
|-------------|---------------|
| Running Gatewerk | `git clone https://github.com/gatewerk/gatewerk && ./scripts/quickstart.sh` — first review in about ten minutes, most of it Docker build time |
| `ANTHROPIC_API_KEY` | Set in your environment before running this notebook |
| `GATEWERK_API_KEY` | Printed by the seed container (`docker compose logs gatewerk-seed`); starts with `gwk_` |
| `requests` library | Installed in the next cell |

After `./scripts/quickstart.sh`:
- Dashboard: http://localhost:8880 (login: admin@gatewerk.local / admin123)
- API: http://localhost:3100

> **Port conflict?** If port 3100 is already in use on your machine, Gatewerk's
> `docker-compose.yml` can be edited to remap it. Update `GATEWERK_BASE_URL` below
> to match the port you chose.

In [ ]:
%pip install anthropic requests --quiet

In [ ]:
import os
import time
import json
import requests
import anthropic

# ── Configuration ──────────────────────────────────────────────────────────────
# Never hardcode credentials. Read from environment variables.
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
GATEWERK_API_KEY  = os.environ.get("GATEWERK_API_KEY")   # starts with gwk_
GATEWERK_BASE_URL = os.environ.get("GATEWERK_BASE_URL", "http://localhost:3100")

# Using claude-haiku-4-5 for cost-efficient tool-use demos.
# Upgrade to claude-sonnet-4-5 or claude-opus-4-5 for production judgment tasks.
MODEL = "claude-haiku-4-5"

# Safety check: halt early with a clear message rather than a confusing API error.
if not ANTHROPIC_API_KEY:
    raise EnvironmentError(
        "ANTHROPIC_API_KEY is not set.\n"
        "Export it before launching Jupyter: export ANTHROPIC_API_KEY=sk-ant-..."
    )
if not GATEWERK_API_KEY:
    raise EnvironmentError(
        "GATEWERK_API_KEY is not set.\n"
        "Run: docker compose logs gatewerk-seed  (look for the gwk_... line)\n"
        "Then: export GATEWERK_API_KEY=gwk_..."
    )

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print(f"Using model : {MODEL}")
print(f"Gatewerk URL: {GATEWERK_BASE_URL}")
print(f"API key     : {GATEWERK_API_KEY[:8]}...")

## Step 1 — Define the Gatewerk tool and run the Claude tool-use loop

We expose a single tool to Claude: `create_gatewerk_review`. Claude decides when
the task needs human approval and calls it. We execute the HTTP call, return the
result, and Claude continues.

The task: Claude drafts a reply email, then requests human review before sending.

**Real API shape** (`POST /api/v1/reviews` — verified against `openapi.snapshot.json`):
- Required: `template` (slug string), `payload` (object)
- Optional: `priority`, `callback_url`, `idempotency_key`, and others
- The seeded `email-review` template expects fields `to`, `subject`, `body`, `tone`

Expected response (HTTP 201): `{ id, status: "pending", decision: null, ... }`

In [ ]:
# ── Tool definition ────────────────────────────────────────────────────────────
# Mirrors the real ReviewCreateBody: required fields are `template` and `payload`.
# There is no `title` field on the API — the template slug identifies the review type.
GATEWERK_TOOL = {
    "name": "create_gatewerk_review",
    "description": (
        "Submit content for human review in Gatewerk. "
        "Call this whenever you need a human to approve, reject, or edit your "
        "output before it is acted upon. Returns a review id; poll it to get "
        "the human's decision once status becomes 'decided'."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "template": {
                "type": "string",
                "description": (
                    "Template slug that determines the review form fields. "
                    "Use 'email-review' for email draft reviews."
                )
            },
            "payload": {
                "type": "object",
                "description": (
                    "The content to review. For email-review, include: "
                    "to (string), subject (string), body (string), tone (string)."
                )
            },
            "priority": {
                "type": "string",
                "enum": ["low", "normal", "high", "urgent"],
                "description": "Review priority. Defaults to 'normal'."
            }
        },
        "required": ["template", "payload"]
    }
}

# ── Execute tool call against Gatewerk ────────────────────────────────────────
def run_create_review(tool_input: dict) -> dict:
    """POST to /api/v1/reviews and return the created review object.

    Body shape matches ReviewCreateBody in openapi.snapshot.json:
      required: template (slug), payload (object)
      optional: priority, callback_url, idempotency_key, ...
    """
    url = f"{GATEWERK_BASE_URL}/api/v1/reviews"
    headers = {
        "Authorization": f"Bearer {GATEWERK_API_KEY}",
        "Content-Type":  "application/json",
    }
    body: dict = {
        "template": tool_input["template"],
        "payload":  tool_input["payload"],
    }
    if "priority" in tool_input:
        body["priority"] = tool_input["priority"]
    resp = requests.post(url, headers=headers, json=body, timeout=10)
    resp.raise_for_status()
    return resp.json()

# ── Claude tool-use loop ───────────────────────────────────────────────────────
TASK_PROMPT = (
    "You are drafting a follow-up email to a user named Alex who asked about our "
    "new pricing. Draft a short, friendly reply (2-3 sentences), then use the "
    "create_gatewerk_review tool to submit the draft for human approval before "
    "it is sent. Use template 'email-review'. Include to, subject, body, and tone "
    "as separate fields in the payload."
)

messages = [{"role": "user", "content": TASK_PROMPT}]
review_id = None

print("Running Claude tool-use loop...\n")

while True:
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        tools=[GATEWERK_TOOL],
        messages=messages,
    )

    # Append assistant turn
    messages.append({"role": "assistant", "content": response.content})

    if response.stop_reason == "end_turn":
        for block in response.content:
            if hasattr(block, "text"):
                print("Claude:", block.text)
        break

    if response.stop_reason == "tool_use":
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"Claude called tool: {block.name}")
                print("  Input:", json.dumps(block.input, indent=2))

                if block.name == "create_gatewerk_review":
                    result = run_create_review(block.input)
                    review_id = result.get("id")
                    print(f"  Review created: {review_id}  (status={result.get('status')!r})")
                    tool_results.append({
                        "type":        "tool_result",
                        "tool_use_id": block.id,
                        "content":     json.dumps(result),
                    })

        messages.append({"role": "user", "content": tool_results})

print(f"\nReview ID to poll: {review_id}")

## Step 2 — Go decide in the Inbox

Open the Gatewerk dashboard and approve, reject, or edit the review.

**Dashboard:** http://localhost:8880 (login: admin@gatewerk.local / admin123)

Navigate to **Inbox**, click the review, then use the action buttons in the panel:
- **Approve** — the email draft is accepted as-is.
- **Reject** — the draft is discarded; Claude will know and can try again.
- **Edit fields then approve** — the edited values return to the agent in `approved_value`.

The poll cell below waits for `status` to become `"decided"` — the real terminal
state. The outcome is in the separate `decision` field (`approved` / `rejected` / `edited`).

Run the next cell once you have made a decision (or leave it running — it polls every 5 s).

In [ ]:
# ── Poll until status == "decided" ─────────────────────────────────────────────
# Gatewerk's terminal state for a decided review is status="decided".
# The outcome (approved/rejected/edited) lives in the separate `decision` field.
# Long-lived agent processes should use webhooks instead; polling keeps the
# notebook self-contained.

if review_id is None:
    raise RuntimeError("No review_id from the previous cell. Run Step 1 first.")

MAX_WAIT_SECONDS = 300  # 5 minutes; extend if you need more time
POLL_INTERVAL    = 5    # seconds between checks

url = f"{GATEWERK_BASE_URL}/api/v1/reviews/{review_id}"
headers = {"Authorization": f"Bearer {GATEWERK_API_KEY}"}

decision_data = None
elapsed = 0

print(f"Polling {url}")
print("Waiting for status='decided' ...")

while elapsed < MAX_WAIT_SECONDS:
    resp = requests.get(url, headers=headers, timeout=10)
    resp.raise_for_status()
    review = resp.json()

    status = review.get("status")
    if status == "decided":
        decision_data = review
        print(f"\nDecision received — decision={review.get('decision')!r}")
        break

    print(f"  status={status!r} — waiting {POLL_INTERVAL}s (elapsed {elapsed}s)...")
    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
else:
    print("Timed out waiting for a decision. Re-run this cell or extend MAX_WAIT_SECONDS.")

if decision_data:
    # Print the fields most relevant to the agent
    print(json.dumps({
        "id":             decision_data.get("id"),
        "status":         decision_data.get("status"),
        "decision":       decision_data.get("decision"),
        "approved_value": decision_data.get("approved_value"),
        "feedback":       decision_data.get("feedback"),
    }, indent=2))

## Step 3 — Branch on the decision

Feed the outcome back to Claude. The `decision` field holds `approved`, `rejected`,
or `edited`. When the reviewer edited the payload, `approved_value` on the review
object is the authoritative corrected content — use it instead of the original payload.

In [ ]:
# ── Branch on decision field ───────────────────────────────────────────────────
# `status` is now "decided"; the outcome is in the separate `decision` field.
# `approved_value` is set by the server for both "approved" and "edited" decisions
# and contains the payload that should proceed.

if decision_data is None:
    raise RuntimeError("No decision_data from the previous cell. Run Step 2 first.")

decision       = decision_data.get("decision")        # "approved" | "rejected" | "edited"
approved_value = decision_data.get("approved_value")   # authoritative payload when set
feedback       = decision_data.get("feedback") or ""

if decision == "approved":
    # approved_value mirrors the original payload when no edits were made
    final_payload = approved_value or decision_data.get("payload", {})
    followup_msg = (
        f"The human approved the email draft without changes. "
        f"Final content ready to send: {json.dumps(final_payload)}. "
        f"Confirm in one sentence that you would now call the send-email function."
    )

elif decision == "edited":
    # Reviewer changed one or more fields — approved_value is the corrected version
    final_payload = approved_value or decision_data.get("payload", {})
    followup_msg = (
        f"The human reviewed and edited the draft. "
        f"Edited content: {json.dumps(final_payload)}. "
        f"Reviewer note: '{feedback}'. "
        f"Confirm in one sentence that you would send this edited version."
    )

elif decision == "rejected":
    final_payload = None
    followup_msg = (
        f"The human rejected the email draft. Reviewer note: '{feedback}'. "
        f"In one sentence, describe what you would do differently in a revised draft."
    )

else:
    raise ValueError(f"Unexpected decision value: {decision!r}")

print(f"Decision       : {decision}")
print(f"Final payload  : {json.dumps(final_payload, indent=2)}")
print(f"Feedback       : {feedback!r}")
print()

# ── Pass outcome back to Claude ────────────────────────────────────────────────
messages.append({"role": "user", "content": followup_msg})

followup = client.messages.create(
    model=MODEL,
    max_tokens=512,
    messages=messages,
)

for block in followup.content:
    if hasattr(block, "text"):
        print("Claude:", block.text)

## What just happened

1. Claude drafted an email and called `create_gatewerk_review` with `template='email-review'`
   and the draft as `payload`. The API returned a review with `status='pending'`.
2. The poll loop called `GET /api/v1/reviews/:id` every 5 seconds until `status` became
   `'decided'` — the real terminal state Gatewerk sets once a human acts.
3. The `decision` field on the decided review held the outcome: `approved`, `rejected`,
   or `edited`.
4. On `edited` (or `approved`), `approved_value` from the review object — not the
   original payload — was used as the authoritative content Claude acts on.

## Next steps

- **Replace polling with webhooks** for production. Pass `callback_url` in the create
  request; Gatewerk fires `review.action_taken` (Standard Webhooks format, HMAC-signed)
  when a decision is recorded.
- **External reviewers** — pass `assignee` (email address) in the create body to send
  a share-link via email-OTP; the reviewer decides without needing a Gatewerk account.
- **MCP for Claude Code** — the `@gatewerk/mcp` package wires the MCP server directly;
  no HTTP client code needed.
- **Other seeded templates**: `proposal-review`, `code-deploy`, `content-approval`,
  `expense-report`, `customer-reply`.

Full docs: https://gatewerk.com/docs

GitHub: https://github.com/gatewerk/gatewerk